# 7 · Time cubes & projections

`timecube` turns a `DatasetCollection` into a map with a **time slider**; `play` adds an animation player
and `save_animation` writes a GIF. `projection` re-renders the map in a non-Mercator projection (through the
matplotlib backend) and `graticule` adds lon/lat grid lines. We use the 12-month WorldClim temperature stack.

**Setup** — Bokeh extension and the 12 monthly global-temperature rasters as a `DatasetCollection`.

In [ ]:
from pathlib import Path

# Resolve the repo root so the bundled sample data is found whether this runs from
# docs/examples/interactive/ (mkdocs) or the repository root.
ROOT = Path.cwd()
while not (ROOT / "examples" / "data" / "LisbonElevation.tif").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "examples" / "data"

import holoviews as hv
hv.extension("bokeh")            # the interactive tier renders through Bokeh

from pyramids.dataset import Dataset
from pyramids.dataset.collection import DatasetCollection
from digitalearth.interactive import InteractiveMap

months = [str(DATA / "global" / f"wc2.1_10m_tavg_{m:02d}.tif") for m in range(1, 13)]
cube = DatasetCollection.from_files(months)
LABELS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

### `timecube` — a time slider
Drag the **time** slider to step through the 12 months. A shared colour range keeps the colormap stable across frames, so the maps are comparable.

In [ ]:
m = InteractiveMap(crs=4326, title="monthly mean temperature")
m.timecube(cube, labels=LABELS, cmap="inferno")
m

### `play` — an animation player
`play` returns a Panel layout with a player widget (▶) over the time cube — press play to animate through the months.

In [ ]:
m = InteractiveMap(crs=4326, title="temperature animation")
m.timecube(cube, labels=LABELS, cmap="inferno")
m.play(fps=4)

### `save_animation` — write a GIF
Render the cube to a GIF on disk (one frame per member). We write to a temporary folder here so the docs tree stays clean.

In [ ]:
import tempfile

m = InteractiveMap(crs=4326)
m.timecube(cube, labels=LABELS, cmap="inferno")
out = m.save_animation(str(Path(tempfile.mkdtemp()) / "temperature.gif"), fps=4)
print("wrote", Path(out).name, "—", Path(out).stat().st_size, "bytes")

### `projection` + `graticule` — non-Mercator
Re-render in an **Orthographic** projection with a graticule. Projected maps go through HoloViews' matplotlib backend (GeoViews reprojects), so we display the rendered object explicitly.

In [ ]:
elev = Dataset.read_file(str(DATA / "global" / "wc2.1_10m_elev.tif"))
m = InteractiveMap(crs=4326, title="orthographic globe")
m.image(elev, cmap="terrain").projection("Orthographic").graticule()
hv.output(m.render(), backend="matplotlib")